# Unit 3, Lecture 5: Multi-agent systems, the graph model

The finale of the unit. In Lecture 3 you saw the door: an agent can be another
agent's tool. Today you walk through it, two ways.

- **The simple way**: a manager agent that delegates to specialist agents.
- **The powerful way**: a **graph**, nodes that do work and edges that route,
  the model that replaced AutoGen's free conversation.

The graph of plain-function nodes runs **offline, no model**, so its whole flow
is testable. The manager needs a lane.

## 1. The simple way: a manager over specialists

Each specialist is an ordinary agent. `as_tool()` turns it into something the
manager can call. **This cell needs a lane.**

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from cse476.agent_fw import make_client
from cse476.multi_agent import build_manager

manager = build_manager(make_client())
print(await manager.run("A customer was charged twice and wants a refund. Handle it."))

The manager delegated to the billing specialist and returned the answer. For
a small team and a simple flow, this is all you need.

**But the plan lives in the manager's head.** When the flow has real structure,
fixed order, branches, work that fans out and rejoins, you want that structure
written down and enforced, not remembered by a model. That is the graph.

## 2. The graph: nodes and edges

Each node is a plain async function decorated as an `@executor`. It does work
and sends a message onward. The edges say what runs next. Read them and you see
the entire flow. **No model runs here.**

In [ ]:
from cse476.multi_agent import build_triage_graph, classify_node, enrich_node, assign_node

graph = build_triage_graph()
print("the graph:", type(graph).__name__)
print()
print("the flow, readable in the builder:")
print("  classify  ->  enrich  ->  assign")
print("  (decide queue) (attach SLA) (produce result)")

## 3. Run it, deterministically, with no model

In [ ]:
from cse476.multi_agent import run_triage_graph

for ticket in [
    "I was charged twice and want a refund",
    "someone hacked my account and is sending spam",
    "the app crashes with an error on login",
]:
    print(f"{ticket[:38]:40} -> {await run_triage_graph(ticket)}")

Same input, same output, every time, and not a token spent. Because the nodes
are plain functions, you can unit test the entire multi-agent flow offline, then
swap any node for a real agent when that step needs judgement. **Structure you
can test, intelligence where you need it.**

## 4. The conversation-to-graph shift

This is the idea to carry into an interview about the framework merger.

In [ ]:
from cse476.multi_agent import conversation_vs_graph

for k, v in conversation_vs_graph().items():
    print(f"{k:22}: {v}")

AutoGen put agents in a room and hoped the conversation converged. Agent
Framework draws an explicit graph you can inspect, test, and reproduce. That is
why the frameworks merged and moved to graphs: a free conversation is powerful
but unreliable; a graph is control flow you can trust.

## 5. The graph, mapped to what you built by hand

In [ ]:
from cse476.multi_agent import GRAPH_MAP

for graph_concept, hand_built in GRAPH_MAP.items():
    print(f"{graph_concept:20} ->  {hand_built}")

Five of the six are your Unit 2 pipeline. Only **fan-out and fan-in**, one
input to many workers and back, are genuinely new, and those are exactly what a
plain pipeline could not express. The graph is your pipeline plus the power it
was missing.

## Your turn

**1. Extend the graph.** Add a fourth node, for example a `notify` step after
`assign`. Notice you add one edge and the flow updates, still fully readable in
the builder.

**2. Make a node an agent.** Replace the `classify` function node with a real
agent that classifies. The rest of the graph does not change. See how structure
and intelligence combine.

**3. Draw before you build.** For your capstone, draw the multi-agent flow as a
graph on paper first. If you cannot draw it, you cannot build it reliably. The
drawing is the design.

In [ ]:
# your work here
